In [7]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

# Define smoothing function using moving average
def moving_average(data, window_size=5):
    return np.convolve(data, np.ones(window_size)/window_size, mode='valid')

# Define smoothing function using Gaussian filter
def gaussian_smooth(data, sigma=2):
    return gaussian_filter1d(data, sigma=sigma)

# Define network architecture with batch normalization and optional dropout
class SimpleNetwork(nn.Module):
    def __init__(self, N_e, N_i, dropout_prob=0.5, use_dropout=True):
        super(SimpleNetwork, self).__init__()
        self.N_e = N_e
        self.N_i = N_i
        self.use_dropout = use_dropout  # Add flag for using dropout
        if self.use_dropout:
            self.dropout = nn.Dropout(dropout_prob)
        self.batch_norm_e = nn.BatchNorm1d(N_e)
        self.batch_norm_i = nn.BatchNorm1d(N_i)
        # Initialize weights
        self.W_e = nn.Parameter(torch.randn(N_e))
        self.W_i = nn.Parameter(torch.randn(N_i))

    def forward(self, I_e, I_i, activation_func='shunting'):
        # Ensure inputs are 2D for batch normalization
        I_e = I_e.unsqueeze(0)  # Add batch dimension
        I_i = I_i.unsqueeze(0)

        # Apply batch normalization only if batch size > 1
        if I_e.size(0) > 1:
            I_e = self.batch_norm_e(I_e)
            I_i = self.batch_norm_i(I_i)

        # Remove batch dimension after normalization
        I_e = I_e.squeeze(0)
        I_i = I_i.squeeze(0)

        # Compute inputs
        excitatory_input = torch.sum(self.W_e * I_e)
        inhibitory_input = torch.sum(self.W_i * I_i)

        if activation_func == 'shunting':
            Y = excitatory_input / (excitatory_input + inhibitory_input + 1)
        elif activation_func == 'relu':
            Y = torch.relu(excitatory_input - inhibitory_input)
        elif activation_func == 'sigmoid':
            Y = torch.sigmoid(excitatory_input - inhibitory_input)
        else:
            raise ValueError("Unknown activation function")

        # Apply dropout during training if enabled
        if self.use_dropout:
            Y = self.dropout(Y)
        return Y

# Define function to compute Rayleigh Quotient (RQ)
def compute_rq(W, X):
    W = W.flatten()  # Ensure W is a 1D array
    covariance_matrix = np.cov(X.T)
    numerator = W @ covariance_matrix @ W.T
    denominator = W @ W.T
    return numerator / denominator

# Function to prune weights based on Rayleigh Quotient (RQ)
def prune_low_rq_weights(W, X, prune_percentage):
    # Compute RQ for the weight vector (W_e or W_i)
    rq_value = compute_rq(W, X)
    threshold = np.percentile(rq_value, prune_percentage)
    mask = rq_value >= threshold
    return W[mask]

# Define training function with pruning
def train_network_with_pruning(N_e, N_i, W_e, W_i, I_e, I_i, target_Y, learning_rate=0.01, epochs=100, activation_func='shunting', lambda_l2=0.0, prune_percentage=0, use_dropout=True):
    # Convert input arrays to tensors
    W_e, W_i, I_e, I_i, target_Y = map(torch.tensor, (W_e, W_i, I_e, I_i, target_Y))
    W_e, W_i = W_e.float(), W_i.float()
    I_e, I_i, target_Y = I_e.float(), I_i.float(), target_Y.float()
    
    # Initialize the network with optional dropout
    net = SimpleNetwork(N_e, N_i, dropout_prob=0.5, use_dropout=use_dropout)
    
    # Define optimizer with L2 regularization (weight decay)
    optimizer = optim.Adam(net.parameters(), lr=learning_rate, weight_decay=lambda_l2)
    
    # Define loss function
    criterion = nn.MSELoss()

    mse_list = []
    
    # Training loop
    for epoch in range(epochs):
        net.train()
        optimizer.zero_grad()

        # Forward pass
        output = net(I_e, I_i, activation_func)
        
        # Compute loss
        loss = criterion(output, target_Y)
        mse_list.append(loss.item())
        
        # Backward pass and optimize
        loss.backward()
        optimizer.step()
        
        # Prune weights based on RQ
        if prune_percentage > 0:
            pruned_W_e = prune_low_rq_weights(net.W_e.data.numpy(), I_e.numpy(), prune_percentage)
            pruned_W_i = prune_low_rq_weights(net.W_i.data.numpy(), I_i.numpy(), prune_percentage)
            net.W_e.data = torch.tensor(pruned_W_e, dtype=torch.float32)
            net.W_i.data = torch.tensor(pruned_W_i, dtype=torch.float32)
        
        if (epoch+1) % 10 == 0:
            print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

    # Calculate final output variance
    net.eval()
    with torch.no_grad():
        outputs = []
        for _ in range(100):  # Sample multiple times to estimate variance
            outputs.append(net(I_e, I_i, activation_func).item())
        final_variance = np.var(outputs)

    return net, mse_list, final_variance

# Function to run the experiment for different pruning ratios
def run_experiment_with_pruning(N_total, W_e, W_i, I_e, I_i, target_Y, learning_rate=0.01, epochs=100, activation_func='shunting', prune_percentage=10, lambda_l2=0.0, use_dropout=True):
    mse_results = []
    variance_results = []
    pruning_ratios = range(0, prune_percentage + 1, 10)  # Test different pruning ratios

    for prune_ratio in pruning_ratios:
        _, mse_list, final_variance = train_network_with_pruning(
            N_e=N_total//2,
            N_i=N_total//2,
            W_e=W_e[:N_total//2],
            W_i=W_i[:N_total//2],
            I_e=I_e[:N_total//2],
            I_i=I_i[:N_total//2],
            target_Y=target_Y,
            learning_rate=learning_rate,
            epochs=epochs,
            activation_func=activation_func,
            prune_percentage=prune_ratio,
            lambda_l2=lambda_l2,
            use_dropout=use_dropout
        )
        mse_results.append(mse_list[-1])  # Store the final MSE
        variance_results.append(final_variance)  # Store the final variance

    # Smooth the results
    mse_smoothed = gaussian_smooth(mse_results, sigma=1)
    variance_smoothed = gaussian_smooth(variance_results, sigma=1)

    # Plot MSE results
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(pruning_ratios, mse_smoothed, label='MSE (Smoothed)')
    plt.xlabel('Pruning Ratio (%)')
    plt.ylabel('Mean Squared Error (MSE)')
    plt.title('MSE vs Pruning Ratio')
    plt.legend()

    # Plot Variance results
    plt.subplot(1, 2, 2)
    plt.plot(pruning_ratios, variance_smoothed, label='Variance (Smoothed)', color='red')
    plt.xlabel('Pruning Ratio (%)')
    plt.ylabel('Variance of Final Output')
    plt.title('Variance vs Pruning Ratio')
    plt.legend()

    plt.tight_layout()
    plt.show()

# Example Usage
N_total = 100
W_e = np.random.uniform(0, 1, N_total)
W_i = np.random.uniform(0, 1, N_total)
I_e = np.random.normal(0.5, 0.1, N_total)
I_i = np.random.normal(0.5, 0.1, N_total)
target_Y = np.array([0.5])  # Example target

run_experiment_with_pruning(N_total, W_e, W_i, I_e, I_i, target_Y, learning_rate=0.00001, epochs=100, activation_func='shunting', prune_percentage=50)

In [ ]:

# Define smoothing function using moving average
def moving_average(data, window_size=5):
    return np.convolve(data, np.ones(window_size)/window_size, mode='valid')

# Define smoothing function using Gaussian filter
def gaussian_smooth(data, sigma=2):
    return gaussian_filter1d(data, sigma=sigma)

